In [1]:
%load_ext autoreload
%autoreload 2

#### I compared the station_name but didn't compare the native_id. While the raws insertaion based on native_id to recongnize station. 

Because I didn't compare native_id before inserting, and the native_id exist many mistach, thus caused some confusion.

It can be mainly devided into two parts, the first batch is different id, same station_name; the second batch is different id, different 


Thus, only the data with same native_id has been inserted and merged; data with different native_id are recognized as different stations.

### Match stations, using station name, lat, lon. The name matching in Raw data meta_data form, in the folder and in the database

In [2]:
from crmprtd import Row
import logging
import os
import pickle
import pandas as pd
import numpy as np
from natsort import natsorted
from natsort import natsort_keygen
from pprint import pprint
from collections import Counter
from collections import defaultdict

import re
from rapidfuzz import fuzz
from crmprtd import infer
from fern_func import *
from sql_func import *
import sqlalchemy as sa
from fern_raw_db_dompare import *

In [3]:
engine = sa.create_engine("postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)

In [23]:
# --- your SQL query ---
query = sa.text("""
    SELECT DISTINCT 
        m.station_name, 
        m.lat,
        m.lon,
        m.elev,
        s.native_id, 
        s.mod_user
    FROM meta_history AS m
    JOIN meta_station AS s 
        ON m.station_id = s.station_id
    WHERE s.network_id = 11;
""")

# --- execute and read into a pandas DataFrame ---
with engine.connect() as conn:
    df = pd.read_sql(query, conn)
# df now contains the result

df_multi = df[df.groupby("station_name")["native_id"].transform("nunique") > 1]
df_multi = df_multi.reset_index(drop=True)

order = {"crmp": 0, "tongli1997": 1}

df_multi = (
    df_multi
    .assign(sort_key=df_multi["mod_user"].map(order).fillna(2))
    .sort_values(by=["station_name", "sort_key"])
    .drop(columns="sort_key")
    .reset_index(drop=True)
)
df_multi

,station_name,lat,lon,elev,native_id,mod_user
0,BulkleyWx,53.771956,-122.720939,599.00,1113694,crmp
1,BulkleyWx,53.772183,-122.720729,NaN,PGTIS1,tongli1997
2,CPFWx,53.753756,-122.718803,595.00,1113682,crmp
3,CPFWx,53.753775,-122.718864,NaN,PGTIS3,tongli1997
4,Canoe Mountain Stn,52.709800,-119.191000,2210.00,Canoe,crmp
5,Canoe Mountain Stn,52.709820,-119.191260,NaN,Canoe Mountain Stn,tongli1997
6,Caribou Pass,58.357180,-129.546370,1744.00,Caribou Pass,crmp
7,Caribou Pass,58.357180,-129.546370,NaN,CaribouPass,tongli1997
8,EP1104.02 Quesnel Highland (ESSFwc3),52.609233,-121.411100,1558.00,Blackbear,crmp
9,EP1104.02 Quesnel Highland (ESSFwc3),52.676367,-121.186733,1621.00,Lower_Grain_Creek,crmp


In [24]:
# --- output folder ---
output_folder = '/workspaces/crmprtd/fern/all_stations_insert/V3/rows_output/'
os.makedirs(output_folder, exist_ok=True)


db_raw_file = pd.read_csv(output_folder + "Fern_raw_db_station_matched_v1.csv")
df_raw = db_raw_file[['station_name', 'native_id', 'lat', 'lon', 'elev']].drop_duplicates().reset_index(drop=True)

In [25]:
raw_elev = df_raw.set_index("native_id")["elev"]
raw_elev

# 2. Function to fill elevation using native_id
def fill_elev(row):
    if pd.isna(row["elev"]):  # only fill missing
        nid = row["native_id"]
        if nid in raw_elev.index:
            return raw_elev.loc[nid]
    return row["elev"]

# 3. Apply the fill
df_multi["elev"] = df_multi.apply(fill_elev, axis=1)
df_multi

,station_name,lat,lon,elev,native_id,mod_user
0,BulkleyWx,53.771956,-122.720939,599.00,1113694,crmp
1,BulkleyWx,53.772183,-122.720729,594.00,PGTIS1,tongli1997
2,CPFWx,53.753756,-122.718803,595.00,1113682,crmp
3,CPFWx,53.753775,-122.718864,593.00,PGTIS3,tongli1997
4,Canoe Mountain Stn,52.709800,-119.191000,2210.00,Canoe,crmp
5,Canoe Mountain Stn,52.709820,-119.191260,2210.00,Canoe Mountain Stn,tongli1997
6,Caribou Pass,58.357180,-129.546370,1744.00,Caribou Pass,crmp
7,Caribou Pass,58.357180,-129.546370,1744.00,CaribouPass,tongli1997
8,EP1104.02 Quesnel Highland (ESSFwc3),52.609233,-121.411100,1558.00,Blackbear,crmp
9,EP1104.02 Quesnel Highland (ESSFwc3),52.676367,-121.186733,1621.00,Lower_Grain_Creek,crmp


In [76]:
import pandas as pd
import numpy as np

# Haversine to compute distance in meters
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # meters
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Pair rows that share the same station_name
pairs = []
for stn, group in df_multi.groupby("station_name"):
    if len(group) < 2:
        continue
    g = group.reset_index()

    # Compare every pair within the station_name group
    for i in range(len(g)):
        for j in range(i+1, len(g)):
            row1 = g.loc[i]
            row2 = g.loc[j]
            pairs.append({
                "station_name": stn,
                "native_id_1": row1["native_id"],
                "native_id_2": row2["native_id"],
                # "index1": row1["index"],
                # "index2": row2["index"],
                "dlat": abs(row1["lat"].round(5) - row2["lat"].round(5)),
                "dlon": abs(row1["lon"].round(5) - row2["lon"].round(5)),
                "delev": abs(row1["elev"] - row2["elev"]),
                "distance_m": haversine(row1["lat"].round(5), row1["lon"].round(5), row2["lat"].round(5), row2["lon"].round(5)),
                "mod_user_1": row1["mod_user"],
                "mod_user_2": row2["mod_user"],
            })

pairs_df = pd.DataFrame(pairs)
df_compare = pairs_df[pairs_df['mod_user_2'] == 'tongli1997'].reset_index(drop = True)

md_string = df_compare.to_markdown(index=False)

# Write to file
with open(output_folder + "station_comparison.md", "w") as f:
    f.write(md_string)

df_compare

,station_name,native_id_1,native_id_2,dlat,dlon,delev,distance_m,mod_user_1,mod_user_2
0,BulkleyWx,1113694,PGTIS1,0.00022,0.00021,5.00,28.087063,crmp,tongli1997
1,CPFWx,1113682,PGTIS3,0.00002,0.00006,2.00,4.528383,crmp,tongli1997
2,Canoe Mountain Stn,Canoe,Canoe Mountain Stn,0.00002,0.00026,0.00,17.656216,crmp,tongli1997
3,Caribou Pass,Caribou Pass,CaribouPass,0.00000,0.00000,0.00,0.000000,crmp,tongli1997
4,EndakoWx,1159701,Endako,0.00011,0.00021,5.00,18.280584,crmp,tongli1997
5,GeorgeWx,1177893,George,0.08034,0.04738,19.00,9456.760499,crmp,tongli1997
6,GunnelWx,1113693,Gunnel1,0.00000,0.00000,308.00,0.000000,crmp,tongli1997
7,Kluskus,Kluskus,SBSmc3Wx,0.00005,0.00007,0.35,7.242236,crmp,tongli1997
8,MiddleforkWx,1095443,MF1,0.00000,0.00000,0.00,0.000000,crmp,tongli1997
9,NondaWx,1113695,Nonda1,0.00000,0.00000,0.00,0.000000,crmp,tongli1997


### Different station_name, different_id, but still the same station

In [ ]:
BoulderWx,Boulder Creek
ChiefLakeWx,ChiefWx
CrystalWx,Crystal Lake
Dennis,Dennis
HourglassWx,Hourglass
MacJxnWx,Mackenzie Jxn
Tamarac,Bednesti

In [46]:
# --- your SQL query ---
query = sa.text("""
    SELECT DISTINCT m.station_name, s.native_id, m.lat, m.lon, m.elev, s.mod_user
    FROM meta_history AS m
    JOIN meta_station AS s ON m.station_id = s.station_id
    WHERE s.network_id = 11;
""")

# --- execute and read into a pandas DataFrame ---
with engine.connect() as conn:
    df_2 = pd.read_sql(query, conn)

df_2

target_names = [
    "Boulder Creek", "BoulderWx",
    "Chief Lake", "ChiefLakeWx",
    "Crystal Lake", "CrystalWx",
    "Hourglass", "HourglassWx",
    "Mackenzie Jxn", "MacJxnWx",
    "BednestiWx", "Tamarac",
]

df_selected = df_2[df_2["station_name"].isin(target_names)]


raw_elev = df_raw.set_index("native_id")["elev"]
raw_elev

# 2. Function to fill elevation using native_id
def fill_elev(row):
    if pd.isna(row["elev"]):  # only fill missing
        nid = row["native_id"]
        if nid in raw_elev.index:
            return raw_elev.loc[nid]
    return row["elev"]

# 3. Apply the fill
df_multi["elev"] = df_multi.apply(fill_elev, axis=1)
df_multi

df_selected

/tmp/ipykernel_2298/2357367344.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected["elev"] = df_selected.apply(fill_elev, axis=1)


,station_name,native_id,lat,lon,elev,mod_user
2,BednestiWx,BednestiTamarac,53.871330,-123.370400,865.000,crmp
4,Boulder Creek,BoulderCr,55.108200,-127.375000,385.101,crmp
5,BoulderWx,Boulder Creek,55.108173,-127.374740,385.000,tongli1997
14,Chief Lake,Chief Lake,54.238210,-122.877500,726.000,crmp
15,ChiefLakeWx,ChiefWx,54.238028,-122.877306,720.000,tongli1997
20,Crystal Lake,CrystalLk,54.423500,-122.628000,744.027,crmp
21,CrystalWx,Crystal Lake,54.423494,-122.627844,744.000,tongli1997
39,Hourglass,Hourglass,55.212700,-120.772000,1131.200,crmp
40,HourglassWx,Hourglass,55.212750,-120.771635,1131.000,tongli1997
47,MacJxnWx,Mackenzie Jxn,55.137858,-122.977243,719.000,tongli1997


In [60]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Compute great-circle distance in meters.
    """
    R = 6371000  # Earth radius in meters

    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)

    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [81]:
# Original df_selected
df = df_selected.copy()

# Extract row 2 (BednestiWx crmp) and row 66 (Tamarac tongli1997)
row2 = df.loc[[2]]
row66 = df.loc[[66]]

# Remove row 66 from original
df_rest = df.drop(index=66)

# New order
new_order = pd.concat(
    [
        row2,
        row66,
        df_rest.drop(index=2)
    ],
    axis=0
)

# Reset index
df_ordered = new_order.reset_index(drop=True)

# Assign pair IDs
df_ordered["pair_id"] = df_ordered.index // 2

# Build compare dataframe
pairs = []

for pid, group in df_ordered.groupby("pair_id"):
    if len(group) != 2:
        continue
        
    row_a = group.iloc[0]
    row_b = group.iloc[1]
    
    pairs.append({
        # "pair_id": pid,
        "station_name_A": row_a["station_name"],
        "station_name_B": row_b["station_name"],
        "native_id_A": row_a["native_id"],
        "native_id_B": row_b["native_id"],
        "user_A": row_a["mod_user"],
        "user_B": row_b["mod_user"],
        "lat_diff": abs(row_a["lat"].round(5) - row_b["lat"].round(5)),
        "lon_diff": abs(row_a["lon"].round(5) - row_b["lon"].round(5)),
        "elev_diff": abs(row_a["elev"] - row_b["elev"]),
        "distance_m": haversine(
            row_a["lat"].round(5), row_a["lon"].round(5),
            row_b["lat"].round(5), row_b["lon"].round(5)
        )
    })

df_compare_2 = pd.DataFrame(pairs)
df_compare_2

md_string = df_compare_2.to_markdown(index=False)

# Write to file
with open(output_folder + "station_comparison.md", "w") as f:
    f.write(md_string)

df_compare_2

,station_name_A,station_name_B,native_id_A,native_id_B,user_A,user_B,lat_diff,lon_diff,elev_diff,distance_m
0,BednestiWx,Tamarac,BednestiTamarac,Bednesti,crmp,tongli1997,0.00000,0.00004,0.000,2.622424
1,Boulder Creek,BoulderWx,BoulderCr,Boulder Creek,crmp,tongli1997,0.00003,0.00026,0.101,16.870824
2,Chief Lake,ChiefLakeWx,Chief Lake,ChiefWx,crmp,tongli1997,0.00018,0.00019,6.000,23.517070
3,Crystal Lake,CrystalWx,CrystalLk,Crystal Lake,crmp,tongli1997,0.00001,0.00016,0.027,10.410282
4,Hourglass,HourglassWx,Hourglass,Hourglass,crmp,tongli1997,0.00005,0.00036,0.200,23.505449
5,MacJxnWx,Mackenzie Jxn,Mackenzie Jxn,MacJxn,tongli1997,crmp,0.00004,0.00024,0.277,15.889472


In [77]:
df_compare

,station_name,native_id_1,native_id_2,dlat,dlon,delev,distance_m,mod_user_1,mod_user_2
0,BulkleyWx,1113694,PGTIS1,0.00022,0.00021,5.00,28.087063,crmp,tongli1997
1,CPFWx,1113682,PGTIS3,0.00002,0.00006,2.00,4.528383,crmp,tongli1997
2,Canoe Mountain Stn,Canoe,Canoe Mountain Stn,0.00002,0.00026,0.00,17.656216,crmp,tongli1997
3,Caribou Pass,Caribou Pass,CaribouPass,0.00000,0.00000,0.00,0.000000,crmp,tongli1997
4,EndakoWx,1159701,Endako,0.00011,0.00021,5.00,18.280584,crmp,tongli1997
5,GeorgeWx,1177893,George,0.08034,0.04738,19.00,9456.760499,crmp,tongli1997
6,GunnelWx,1113693,Gunnel1,0.00000,0.00000,308.00,0.000000,crmp,tongli1997
7,Kluskus,Kluskus,SBSmc3Wx,0.00005,0.00007,0.35,7.242236,crmp,tongli1997
8,MiddleforkWx,1095443,MF1,0.00000,0.00000,0.00,0.000000,crmp,tongli1997
9,NondaWx,1113695,Nonda1,0.00000,0.00000,0.00,0.000000,crmp,tongli1997


In [83]:
output_file = output_folder+ "5.different_native_id_station_comparison.xlsx"

with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
    df_compare.to_excel(writer, sheet_name="Same_station_name", index=False)
    df_compare_2.to_excel(writer, sheet_name="Different_station_name", index=False)

In [79]:
!pip install xlsxwriter